In [2]:
import json
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

# Configuration

INPUT_FILE = "gov_data_be_clean.json"

OUTPUT_JSON = "gov_data_be_normalized.json"
OUTPUT_CSV = "domain_distribution_normalized.csv"
OUTPUT_PDF = "domain_distribution_normalized.pdf"

# Publication-quality plotting settings

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
})

# Domain normalization mapping

DOMAIN_MAPPING = {

    # Formatting
    "HEALTH": "Health",
    "Health": "Health",

    "ENVIRONMENT": "Environment",
    "Environment": "Environment",

    "TRANSPORT": "Transport",
    "Transport": "Transport",

    "Government and public sector":
        "Government and Public Sector",
    "GOVERNMENT, PUBLIC SECTOR":
        "Government and Public Sector",

    # Semantic merges
    "SCIENCE AND TECHNOLOGY":
        "Science and Technology",
    "Technology":
        "Science and Technology",

    "POPULATION AND SOCIAL CONDITIONS":
        "Population and Society",
    "Population and society":
        "Population and Society",

    "REGIONS, CITIES":
        "Regions and Cities",
    "Regional":
        "Regions and Cities",

    "JUSTICE, LEGAL SYSTEM, PUBLIC SAFETY":
        "Justice and Public Safety",
    "Justice":
        "Justice and Public Safety",

    "EDUCATION":
        "Education, Culture and Sport",
    "Education, culture and sport":
        "Education, Culture and Sport",

    "Agriculture":
        "Agriculture, Fisheries, Forestry and Food",
    "AGRICULTURE, FISHERIES, FORESTRY, FOOD":
        "Agriculture, Fisheries, Forestry and Food",
}


# domain normalization

def normalize_domain(domain):
    """
    Normalize a domain label using the mapping.
    Domains not present in the mapping are left unchanged.
    """
    domain = domain.strip()
    return DOMAIN_MAPPING.get(domain, domain)

# Load dataset

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"\nTotal datasets loaded: {len(data)}")

# Collect original domain labels

original_domains = set()

for record in data:
    for domain in record.get("domains", []):
        original_domains.add(domain.strip())

# Normalize domains

all_domains = []

for record in data:

    normalized_domains = [
        normalize_domain(domain)
        for domain in record.get("domains", [])
    ]

    # Remove duplicates introduced by normalization
    normalized_domains = list(dict.fromkeys(normalized_domains))

    record["domains"] = normalized_domains

    all_domains.extend(normalized_domains)

# Save normalized dataset

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(
        data,
        f,
        ensure_ascii=False,
        indent=2
    )

# Report unmapped domains

unmapped_domains = sorted(
    d for d in original_domains
    if d not in DOMAIN_MAPPING
)

print("\n========== UNMAPPED DOMAINS ==========\n")

if len(unmapped_domains) == 0:
    print("None")
else:
    for domain in unmapped_domains:
        print(domain)

# Domain statistics

domain_counts = Counter(all_domains)

df = (
    pd.DataFrame(
        domain_counts.items(),
        columns=["domain", "dataset_count"]
    )
    .sort_values(
        by="dataset_count",
        ascending=False
    )
    .reset_index(drop=True)
)

# Add percentages
df["percentage"] = (
    df["dataset_count"]
    / df["dataset_count"].sum()
    * 100
).round(2)

# Save CSV
df.to_csv(OUTPUT_CSV, index=False)

# Console report

print("\n========== DOMAIN DISTRIBUTION ==========\n")
print(df[["domain", "dataset_count"]])

print("\n========== SUMMARY ==========\n")
print(f"Total datasets: {len(data)}")
print(f"Unique domains: {len(df)}")
print(f"Total domain assignments: {df['dataset_count'].sum()}")

print("\n========== TOP 10 DOMAINS ==========\n")
print(df.head(10))

print("\n========== DOMAIN PERCENTAGES ==========\n")
print(df[["domain", "dataset_count", "percentage"]])

print(f"\nNormalized JSON saved to: {OUTPUT_JSON}")
print(f"CSV summary saved to: {OUTPUT_CSV}")

# Visualization

fig, ax = plt.subplots(figsize=(12, 7))

bars = ax.barh(
    df["domain"],
    df["dataset_count"]
)

# Annotate bars with counts

offset = max(df["dataset_count"]) * 0.01

for bar in bars:

    width = bar.get_width()

    ax.text(
        width + offset,
        bar.get_y() + bar.get_height() / 2,
        f"{int(width)}",
        va="center",
        fontsize=11
    )

ax.set_xlabel("Number of datasets")
ax.set_ylabel("Domain")
ax.set_title("Dataset Distribution by Normalized Domain")

ax.invert_yaxis()

# Add space for labels on the right

ax.set_xlim(
    0,
    max(df["dataset_count"]) * 1.15
)

plt.tight_layout()

plt.savefig(
    OUTPUT_PDF,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.05
)

plt.close()

print(f"Vector graph saved to: {OUTPUT_PDF}")

# Additional summary statistics

print("\n========== STATISTICS ==========\n")

print(
    f"Mean datasets per domain: "
    f"{df['dataset_count'].mean():.2f}"
)

print(
    f"Median datasets per domain: "
    f"{df['dataset_count'].median():.2f}"
)

print(
    f"Largest domain: "
    f"{df.iloc[0]['domain']} "
    f"({df.iloc[0]['dataset_count']} datasets)"
)

print(
    f"Smallest domain: "
    f"{df.iloc[-1]['domain']} "
    f"({df.iloc[-1]['dataset_count']} datasets)"
)


Total datasets loaded: 2254

========== UNMAPPED DOMAINS ==========

Economy
Energy
INTERNATIONAL ISSUES

========== DOMAIN DISTRIBUTION ==========

                                       domain  dataset_count
0                      Science and Technology            875
1                                 Environment            632
2                      Population and Society            356
3                                      Health            267
4                                     Economy            257
5                                   Transport            137
6                Government and Public Sector            135
7                                      Energy            116
8                        INTERNATIONAL ISSUES             79
9   Agriculture, Fisheries, Forestry and Food             64
10               Education, Culture and Sport             61
11                         Regions and Cities             46
12                  Justice and Public Safety            